# 🌊 FloodAI — Urban Flood Risk Assessment & Early Warning System

> **⚠️ DISCLAIMER:** This notebook uses a **synthetic** dataset for educational/research purposes only.
> It is NOT an official government flood-warning system.

**SDG Alignment:** SDG 6 (Clean Water) | SDG 11 (Sustainable Cities) | SDG 13 (Climate Action)

## 1. Setup & Imports

In [ ]:
import os, sys, warnings
warnings.filterwarnings('ignore')

# Ensure project root is on path
PROJECT_ROOT = os.path.abspath('..')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)
print('Environment ready ✓')

## 2. Load Dataset

In [ ]:
DATA_PATH = os.path.join(PROJECT_ROOT, 'data', 'raw', 'flood_risk_dataset_india.csv')
df = pd.read_csv(DATA_PATH)
print(f'Shape: {df.shape}')
df.head()

## 3. Dataset Validation

In [ ]:
from src.preprocessing import validate_dataset
report = validate_dataset(df)

print('=== DATASET REPORT ===')
print(f'Rows      : {report["n_rows"]}')
print(f'Columns   : {report["n_cols"]}')
print(f'Missing   : {report["total_missing"]}')
print(f'Duplicates: {report["duplicate_rows"]}')
print()
print('Target distribution:')
for k, v in report.get('target_distribution', {}).items():
    print(f'  {k}: {v}')

In [ ]:
print('Data Types:')
print(df.dtypes)
print()
print('Missing per column:')
print(df.isnull().sum())

## 4. Data Cleaning

In [ ]:
from src.preprocessing import clean_data
df_clean = clean_data(df)
print(f'After cleaning: {df_clean.shape}')

## 5. Exploratory Data Analysis

In [ ]:
TARGET = 'Flood Occurred'
# Target distribution
fig = px.pie(df_clean, names=TARGET, title='Flood vs Non-Flood Distribution',
             color_discrete_sequence=['#3b82d4', '#e74c3c'])
fig.show()

In [ ]:
# Numerical distributions
num_cols = ['Rainfall (mm)', 'Temperature (°C)', 'Humidity (%)',
            'River Discharge (m³/s)', 'Water Level (m)', 'Elevation (m)', 'Population Density']
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()
for i, col in enumerate(num_cols):
    for v, c in [(0, '#3b82d4'), (1, '#e74c3c')]:
        axes[i].hist(df_clean[df_clean[TARGET]==v][col].dropna(), bins=40, alpha=0.6,
                     color=c, label='Flood' if v else 'No Flood')
    axes[i].set_title(col, fontsize=9)
    axes[i].legend(fontsize=7)
axes[-1].set_visible(False)
plt.suptitle('Feature Distributions by Flood Occurrence')
plt.tight_layout()
plt.show()

In [ ]:
# Correlation matrix
corr = df_clean.select_dtypes(include=[np.number]).corr()
fig = px.imshow(corr, color_continuous_scale='RdBu_r', zmin=-1, zmax=1,
                text_auto='.2f', title='Feature Correlation Matrix', aspect='auto')
fig.update_layout(height=600)
fig.show()

In [ ]:
# Geographic scatter
fig = px.scatter_mapbox(
    df_clean.sample(3000, random_state=42),
    lat='Latitude', lon='Longitude',
    color=TARGET,
    color_continuous_scale='RdYlGn_r',
    mapbox_style='open-street-map',
    zoom=3.5, center={'lat': 22, 'lon': 82},
    title='Geographic Distribution of Flood Occurrences',
    height=500,
)
fig.show()

## 6. Feature Engineering

In [ ]:
from src.feature_engineering import engineer_features
df_eng = engineer_features(df_clean)
new_cols = [c for c in df_eng.columns if c not in df_clean.columns]
print('Engineered features:', new_cols)
df_eng[new_cols].describe()

## 7. Preprocessing Pipeline

In [ ]:
from src.preprocessing import get_feature_lists, build_preprocessor, TARGET_COL

X = df_eng.drop(columns=[TARGET_COL])
y = df_eng[TARGET_COL].astype(int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

num_feats, cat_feats = get_feature_lists(X_train)
print(f'Numerical features ({len(num_feats)}): {num_feats}')
print(f'Categorical features ({len(cat_feats)}): {cat_feats}')

## 8. Model Training & Comparison

In [ ]:
import os, json
REPORTS_DIR = os.path.join(PROJECT_ROOT, 'outputs', 'reports')
metrics_path = os.path.join(REPORTS_DIR, 'model_metrics.json')
if os.path.exists(metrics_path):
    with open(metrics_path) as f:
        metrics = json.load(f)
    print(f'Best model: {metrics["best_model"]}')
    rows = []
    for m in metrics['models']:
        rows.append({k: round(v,4) if isinstance(v,float) else v
                     for k,v in m.items()})
    pd.DataFrame(rows).set_index('model')
else:
    print('Run python run_training.py first to generate metrics')

## 9. Model Evaluation — Visualisations

In [ ]:
import joblib
MODELS_DIR = os.path.join(PROJECT_ROOT, 'models')
FIGURES_DIR = os.path.join(PROJECT_ROOT, 'outputs', 'figures')

best_path = os.path.join(MODELS_DIR, 'best_model_pipeline.pkl')
if os.path.exists(best_path):
    pipeline = joblib.load(best_path)
    y_pred = pipeline.predict(X_test)
    y_prob = pipeline.predict_proba(X_test)[:, 1]
    print('Classification Report:')
    print(classification_report(y_test, y_pred, target_names=['No Flood', 'Flood']))
    print(f'ROC-AUC: {roc_auc_score(y_test, y_prob):.4f}')
else:
    print('No trained model found. Run run_training.py first.')

In [ ]:
# Show saved figures
from IPython.display import Image, display
for fname in ['model_comparison.png', 'roc_curves_all.png', 'pr_curves_all.png']:
    path = os.path.join(FIGURES_DIR, fname)
    if os.path.exists(path):
        print(fname)
        display(Image(path))

## 10. Risk Score Analysis

In [ ]:
if os.path.exists(best_path):
    from src.prediction import predict_batch, probability_to_category
    pred_df = predict_batch(pipeline, X_test)
    pred_df['Flood Occurred'] = y_test.values
    
    cat_counts = pred_df['Risk_Category'].value_counts()
    fig = px.pie(values=cat_counts.values, names=cat_counts.index,
                 title='Predicted Risk Category Distribution',
                 color=cat_counts.index,
                 color_discrete_map={'Low':'#16a34a','Moderate':'#d97706',
                                     'High':'#dc2626','Very High':'#7f1d1d'})
    fig.show()
    
    print('Risk Category Counts:')
    print(cat_counts)

## 11. Explainable AI (SHAP)

In [ ]:
if os.path.exists(best_path):
    from src.explainability import compute_shap_values
    shap_vals, X_t, feat_names = compute_shap_values(pipeline, X_test, max_samples=200)
    print(f'SHAP values shape: {shap_vals.shape}')
    print(f'Features: {feat_names[:5]} ...')

In [ ]:
if os.path.exists(best_path):
    import shap
    mean_abs = abs(shap_vals).mean(axis=0)
    fi_df = pd.DataFrame({'feature': feat_names, 'importance': mean_abs})
    fi_df = fi_df.sort_values('importance', ascending=False).head(15)
    fig = px.bar(fi_df, x='importance', y='feature', orientation='h',
                 title='Global Feature Importance (Mean |SHAP|)',
                 color='importance', color_continuous_scale='Blues')
    fig.update_layout(yaxis={'categoryorder':'total ascending'})
    fig.show()

## 12. Geographic Risk Visualisation

In [ ]:
PRED_PATH = os.path.join(PROJECT_ROOT, 'data', 'processed', 'test_predictions.csv')
if os.path.exists(PRED_PATH):
    map_df = pd.read_csv(PRED_PATH)
    fig = px.scatter_mapbox(
        map_df.sample(min(2000, len(map_df)), random_state=42),
        lat='Latitude', lon='Longitude',
        color='Risk_Category',
        color_discrete_map={'Low':'#16a34a','Moderate':'#d97706',
                            'High':'#dc2626','Very High':'#7f1d1d'},
        mapbox_style='open-street-map',
        zoom=3.5, center={'lat': 22, 'lon': 82},
        title='Predicted Flood Risk Map',
        height=550,
    )
    fig.show()
else:
    print('Run run_training.py first to generate predictions')

## 13. Conclusions

### Key Findings

1. **Model Performance:** All three models (LR, RF, XGBoost) achieve meaningful flood prediction accuracy on the synthetic dataset. XGBoost and Random Forest generally outperform Logistic Regression on AUC.

2. **Important Features:** Based on SHAP analysis, features related to rainfall, water level, and river discharge tend to be the strongest predictors of flood occurrence.

3. **Class Imbalance:** The dataset is approximately balanced, but class-weighting techniques were applied as good practice.

4. **Engineered Features:** Interaction terms (Rainfall × Water Level) and severity indicators improve model performance.

### Limitations

- **Synthetic data:** Results cannot be directly applied to real flood prediction.
- **No temporal dimension:** Real flood forecasting requires time-series modelling.
- **No satellite or DEM data:** Real systems use remote sensing inputs.

### SDG Alignment

| SDG | How FloodAI contributes |
|-----|------------------------|
| SDG 6 | Supports water risk awareness |
| SDG 11 | Enables urban flood risk mapping |
| SDG 13 | Demonstrates climate-data-driven decision support |

> ⚠️ This is a **synthetic dataset demonstration project** only.